## 0. Title, metadata, environment

### What

This notebook studies and implements the **ID3** decision-tree algorithm from scratch for **Connect-4 outcome classification** (UTS 32513 Assignment 2, **Option 1**).

| Field | Value |
|-------|--------|
| Project | ID3 for Connect-4 (first-player win / loss / draw) |
| Dataset | [UCI Connect-4](https://archive.ics.uci.edu/dataset/26/connect+4) (Tromp, 1995), `ucimlrepo` id=26 |
| Core libraries | `numpy`, `pandas`, `matplotlib`, `scipy` |
| Reference only | `scikit-learn` — used **only** in §9 for comparison, not for the submitted ID3 model |
| Student name | *[Your name]* |
| Student ID | *[Your student ID]* |

**Citation:** Tromp, J. (1995). Connect-4 [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C59P43.

### Why

A2 requires a **self-contained, public Colab notebook**: markers must be able to run setup → data → train → evaluate without external private files. Declaring allowed libraries, fixing a global random seed, and noting that sklearn is reference-only keeps the submission reproducible and aligned with the Option 1 library policy.

In [ ]:
# How — environment setup and reproducibility
# Core stack for Option 1. scikit-learn is installed for §9 reference comparison only.
# Uncomment the next line when running on a fresh Colab runtime:
# %pip install -q numpy pandas matplotlib scipy ucimlrepo scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"RANDOM_SEED = {RANDOM_SEED}")
print(f"numpy {np.__version__}, pandas {pd.__version__}")
print("Environment ready. Core ID3 uses numpy/pandas only; sklearn appears only in §9.")


## 1. Problem definition (Criterion A)

### What

**Task:** Multi-class classification of the **game-theoretic outcome for the first player** given a legal 8-ply Connect-4 board (neither player has already won; the next move is not forced).

| Role | Specification |
|------|----------------|
| **Training input** | One row of **42** categorical features, each in `{x, o, b}`, for cells `a1`…`g6` |
| **Training output** | Discrete label in `{win, loss, draw}` (value for the first player) |
| **Deployment input** | Same 42-cell schema for one board (or a batch of boards) |
| **Deployment output** | Predicted class; optionally leaf class frequencies as a simple confidence / distribution |

Feature values: `x` = first player, `o` = second player, `b` = blank. Board columns are `a`–`g`, rows `1`–`6` (bottom to top).

### Why

Criterion A needs **technical training and deployment I/O**, not a vague “predict Connect-4” statement. Option 1 also expects a short **historical framing**: why symbolic, information-gain trees were studied for discrete structured domains before continuous parametric models dominated. A clear research question then sets up the evaluation narrative in later sections.

### How

No model code in this section — only the problem contract markers (and later journal readers) need.

**Board layout (columns `a`–`g`, rows `1`–`6` bottom → top):**

```
  6  a6 b6 c6 d6 e6 f6 g6
  5  a5 b5 c5 d5 e5 f5 g5
  4  a4 b4 c4 d4 e4 f4 g4
  3  a3 b3 c3 d3 e3 f3 g3
  2  a2 b2 c2 d2 e2 f2 g2
  1  a1 b1 c1 d1 e1 f1 g1
     a  b  c  d  e  f  g
```

Each cell holds one of `{x, o, b}`. A training example is the flat 42-tuple `(a1,…,g6)` with label in `{win, loss, draw}`.

**Historical motivation (Option 1):** Quinlan’s ID3 selects attributes by **information gain**, inducing a symbolic decision tree over discrete attribute–value data. The hypothesis space is the set of categorical trees—not a continuous weight vector \(\theta\). Discrete board cells with small arity (`{x, o, b}`) match classic multiway ID3 without continuous discretization, which is why Connect-4 is a natural Option 1 case study.

**Research question:** How does greedy information-gain splitting on board-cell attributes behave under class imbalance (and, in later probes, attribute noise), relative to a majority-class baseline and an entropy-based sklearn tree with matched depth limits?

## 2. Learning-theory framing

### What

| Concept | Statement for this notebook |
|---------|------------------------------|
| **Hypothesis space \(H\)** | Finite set of **categorical decision trees** over the 42 cell attributes. Each internal node tests one attribute and branches on values in `{x, o, b}`. \(H\) is **not** parametrized by a continuous weight vector \(\theta\). |
| **Inductive procedure** | ID3 selects one \(h \in H\) by **recursive information-gain (IG) maximization** — a **greedy search** over attributes at each node. |
| **Not gradient descent** | There is no iterative optimizer updating \(\theta\). Training is discrete attribute selection and partition of the sample set \(S\). |
| **Surrogate vs task objective** | **Training criterion:** information gain (reduce conditional entropy of \(Y\) given a split). **Evaluation criteria:** 0–1 accuracy, macro-F1, and confusion structure on held-out boards. These are related but **not identical** objectives. |

### Why

Option 1 asks for **learning-theory depth**, not only “we grew a tree.” Naming \(H\), stating that induction is **greedy IG search** (not gradient descent), and separating the **splitting surrogate** from the **task metrics** sets up Criterion B (algorithm as search in \(H\)) and Criterion C (critical evaluation of IG vs 0–1 / F1). Later sections reuse this framing when we regularize \(H\) with depth / min-sample stops and when we interpret evaluation results.

### How

No training code in this section — only the conceptual pipeline markers need before §5–§8.

**Induction path (this notebook):**

```text
Connect-4 boards (categorical X, label Y)
        │
        ▼
  Choose attribute A* = argmax_A IG(S, A)     ← greedy search in H
        │
        ▼
  Multiway split on {x, o, b}; recurse
        │
        ▼
  Hypothesis h ∈ H  (symbolic tree)
        │
        ▼
  Predict Ŷ on held-out boards
        │
        ▼
  Score with accuracy / macro-F1 / confusion   ← task objectives (≠ IG)
```

**Contrast with parametric learning:** a neural net would pick \(\theta\) by descending a differentiable loss. ID3 never forms \(\theta\); at each node it **enumerates** candidate attributes and keeps the one with highest information gain.

**Criterion C hook (to revisit in §8):** maximizing IG on the training partition does **not** guarantee maximal held-out accuracy or macro-F1—especially under class imbalance (`win` ≫ `draw`). We train with IG; we **judge** the model with task metrics and a majority baseline.

## 3. Data acquisition and preprocessing

### What

| Step | Action |
|------|--------|
| **Acquire** | Fetch UCI Connect-4 (`ucimlrepo` id=26): 42 cell features + `class`. |
| **EDA (lean)** | Class distribution; blank-rate histogram; 6×7 occupancy heatmap — on the **full** data before subsample. |
| **Scale control** | Stratified subsample of size `SUBSAMPLE_N`; bar chart of full vs subsample class counts. |
| **Split** | Stratified **60% / 20% / 20%** train / validation / test; class-composition bars per split. |

**Stated for the study (not checked in code):** features are discrete `{x, o, b}`; target is `{win, loss, draw}`; no missing values; rows treated as i.i.d. positions. Encoding for our tree: keep cell values as strings.

### Why

A2 needs a **self-contained data path** (download → split in one notebook). A short **EDA before subsample** makes class imbalance and board occupancy visible without turning the notebook into an EDA slideshow. Full Connect-4 is large for unrestricted recursive ID3 in Colab, so a **named stratified subsample** is an experimental control—not a substitute for implementing the algorithm. A held-out **validation** set lets §7 tune depth / min-samples without peeking at test.

In [ ]:
# How — load Connect-4 (full table first; subsample + split come after EDA)
from ucimlrepo import fetch_ucirepo

SUBSAMPLE_N = 8000
TARGET_COL = "class"
BOARD_COLS = list("abcdefg")
BOARD_ROWS = list("123456")

repo = fetch_ucirepo(id=26)
df_full = repo.data.features.copy()
df_full[TARGET_COL] = repo.data.targets.iloc[:, 0].astype(str).values
FEATURE_COLS = [c for c in df_full.columns if c != TARGET_COL]

print(f"Loaded: {len(df_full)} rows, {len(FEATURE_COLS)} features")
print(df_full[TARGET_COL].value_counts())


In [ ]:
# How — lean EDA on the FULL dataset (before subsample)

# 1) Class distribution
class_counts = df_full[TARGET_COL].value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(class_counts.index.astype(str), class_counts.values, color="steelblue")
ax.set_title("Class distribution (full Connect-4)")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

# 2) Blank-rate histogram: fraction of cells that are 'b' per board
blank_rate = (df_full[FEATURE_COLS] == "b").sum(axis=1) / len(FEATURE_COLS)
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(blank_rate, bins=20, color="steelblue", edgecolor="white")
ax.set_title("Blank-rate per board (full data)")
ax.set_xlabel("fraction of cells that are blank")
ax.set_ylabel("boards")
plt.tight_layout()
plt.show()

# 3) 6x7 occupancy heatmap: fraction of boards where cell is occupied (not blank)
# Feature names are like a1..g6 → row index 0..5 (row 1 at bottom), col index 0..6
occ = np.zeros((6, 7))
for r_i, r in enumerate(BOARD_ROWS):
    for c_i, c in enumerate(BOARD_COLS):
        col_name = f"{c}{r}"
        occ[5 - r_i, c_i] = (df_full[col_name] != "b").mean()  # row 6 on top for display

fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(occ, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(7))
ax.set_xticklabels(BOARD_COLS)
ax.set_yticks(range(6))
ax.set_yticklabels(list(reversed(BOARD_ROWS)))
ax.set_title("Occupancy rate by cell (full data)")
fig.colorbar(im, ax=ax, fraction=0.046, label="P(cell occupied)")
plt.tight_layout()
plt.show()


In [ ]:
# How — stratified subsample, then full vs subsample class bars
frac = SUBSAMPLE_N / len(df_full)
parts = []
for _, g in df_full.groupby(TARGET_COL):
    parts.append(g.sample(n=int(round(len(g) * frac)), random_state=RANDOM_SEED))
df = pd.concat(parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

full_vc = df_full[TARGET_COL].value_counts()
sub_vc = df[TARGET_COL].value_counts()
labels = full_vc.index.astype(str)
x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(x - w / 2, full_vc.loc[labels].values, w, label="full", color="steelblue")
ax.bar(x + w / 2, sub_vc.reindex(labels).fillna(0).values, w, label="subsample", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title(f"Class counts: full vs subsample (SUBSAMPLE_N={SUBSAMPLE_N})")
ax.set_ylabel("count")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Subsample size: {len(df)}")
print(sub_vc)

In [ ]:
# How — stratified 60/20/20 split + class composition by split
train_parts, val_parts, test_parts = [], [], []
for _, g in df.groupby(TARGET_COL):
    g = g.sample(frac=1.0, random_state=RANDOM_SEED)
    n = len(g)
    i, j = int(0.60 * n), int(0.80 * n)
    train_parts.append(g.iloc[:i])
    val_parts.append(g.iloc[i:j])
    test_parts.append(g.iloc[j:])

train_df = pd.concat(train_parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
val_df = pd.concat(val_parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
test_df = pd.concat(test_parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

split_counts = pd.DataFrame({
    "train": train_df[TARGET_COL].value_counts(),
    "val": val_df[TARGET_COL].value_counts(),
    "test": test_df[TARGET_COL].value_counts(),
}).fillna(0).astype(int)

print(f"Sizes — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
print(split_counts)

split_props = split_counts.div(split_counts.sum(axis=0), axis=1)
ax = split_props.T.plot(kind="bar", stacked=True, figsize=(6, 3.5), colormap="tab20")
ax.set_title("Class composition by split (proportions)")
ax.set_ylabel("proportion")
ax.set_xlabel("split")
ax.legend(title="class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 4. Connect-4 mini-slice warm-up

### What

A **12-board** stratified slice of the training set used only to check entropy and information gain **before** growing a full tree.

| Role | Detail |
|------|--------|
| **Purpose** | Transparent hand-check of \(H(S)\) and \(IG(S,A)\) on Connect-4 labels |
| **Slice** | 4 boards per class (`win` / `loss` / `draw`) from `train_df` |
| **Attribute checked** | `g1` ∈ `{x, o, b}` (multiway split, same arity as §5) |
| **Checkpoint** | Hand numbers must match the `entropy` / `information_gain` routines below |


### Why

Full Connect-4 is too large to verify by hand. A tiny **same-schema** slice lets us write \(H(S)\) and \(IG(S,\texttt{g1})\) on paper, then confirm the **same** formulas in code—without switching to a different toy domain. That math ↔ implementation link is the Option 1 checkpoint before §5.


### How — hand calculation (`g1` on the mini-slice)

With `MINI_SEED = 11` and 4 boards per class, the slice has \(n=12\) and labels **4 / 4 / 4**.

**Root entropy:**

\[
H(S) = -\sum_{c} \frac{4}{12}\log_2\frac{4}{12} = \log_2 3 \approx 1.585
\]

**Partitions by `g1`:**

| `g1` | win | loss | draw | \(n_v\) | \(H(S_v)\) |
|------|-----|------|------|---------|------------|
| `b` | 2 | 1 | 0 | 3 | \(\approx 0.918\) |
| `o` | 1 | 2 | 4 | 7 | \(\approx 1.379\) |
| `x` | 1 | 1 | 0 | 2 | \(1\) |

**Remainder and gain:**

\[
H(S\mid\texttt{g1}) = \frac{3}{12}(0.918) + \frac{7}{12}(1.379) + \frac{2}{12}(1) \approx 1.201
\]

\[
IG(S,\texttt{g1}) = H(S) - H(S\mid\texttt{g1}) \approx 1.585 - 1.201 = 0.384
\]

(IG on 12 rows is only a formula check, not a model: `g1 = o` isolates all four draws but still mixes in wins and losses.) The next cell builds the slice from `train_df` and asserts agreement.


In [ ]:
# How — Connect-4 mini-slice + entropy / IG check
MINI_SEED, CHECK_ATTR = 11, "g1"

mini_df = (
    train_df.groupby(TARGET_COL, group_keys=False)
    .sample(n=4, random_state=MINI_SEED)
    .sample(frac=1.0, random_state=MINI_SEED)
    .reset_index(drop=True)
)

def entropy(y):
    p = pd.Series(y).value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def information_gain(col, y):
    y = pd.Series(y)
    rem = sum(len(g) / len(y) * entropy(g) for _, g in y.groupby(col))
    return entropy(y) - rem

y = mini_df[TARGET_COL]
h_s, ig = entropy(y), information_gain(mini_df[CHECK_ATTR], y)

print(mini_df[[CHECK_ATTR, TARGET_COL]])
print(f"H(S)={h_s:.3f} (hand ~1.585),  IG(g1)={ig:.3f} (hand ~0.384)")
assert abs(h_s - 1.585) < 5e-3 and abs(ig - 0.384) < 5e-3
print("OK — matches hand calculation.")


## 5. Core ID3 algorithm (Criterion B)

### What

From-scratch **multiway ID3** over categorical cells `{x, o, b}`. Reuses `entropy` / `information_gain` from §4.

| Theory | Function |
|--------|----------|
| \(H(S)\), \(IG(S,A)\) | `entropy`, `information_gain` (§4) |
| \(\arg\max_A IG(S,A)\) | `best_split` |
| Recursive induction | `id3_train` → nested dict tree |
| Inference | `predict_one` / `predict` |
| Leaf label | Majority class (`value_counts().idxmax()`) |

**Stops:** pure labels; no attributes left; `max_depth`; `min_samples_leaf`; zero gain. Missing branch at predict → node `default` (majority at that node).


### Why

Criterion B needs the **algorithm as greedy search in \(H\)**, not a library fit. Naming each step (`best_split` → recurse → leaf) maps symbols to code for the journal and Q&A. `max_depth` / `min_samples_leaf` shrink \(H\) so Colab stays runnable on 42 attributes; they are part of the study, not optional polish.


### How — formulas

**Best attribute** at sample set \(S\) with remaining attributes \(\mathcal{A}\):

\[
A^* = \arg\max_{A \in \mathcal{A}} IG(S, A)
\]

**Recursion:** split \(S\) into \(S_v = \{x \in S : A^*=v\}\) for each observed \(v \in \{x,o,b\}\); remove \(A^*\) from \(\mathcal{A}\); recurse. **Leaf:** if \(S\) is pure, or stopping applies → majority label in \(S\).

**Predict:** follow the child’s branch for the row’s attribute value; if that value was never seen at the node, return the node’s stored majority.


In [ ]:
# How — ID3: best_split, train, predict (reuses entropy / information_gain from §4)
# Each node stores n, entropy H(S); decision nodes also store ig for the chosen split.

def majority_label(y):
    # Leaf / fallback label: most frequent class (ties → first in value_counts order)
    return pd.Series(y).value_counts().idxmax()


def best_split(X, y, attributes):
    # Greedy step: pick A* = argmax_A IG(S, A) over remaining attributes
    best_attr, best_ig = attributes[0], -1.0
    for a in attributes:
        ig = information_gain(X[a], y)
        if ig > best_ig:
            best_attr, best_ig = a, ig
    return best_attr, best_ig


def id3_train(X, y, attributes, max_depth=None, min_samples_leaf=1, depth=0):
    # Grow a nested-dict tree: {"attr", "children", "default", ...} or {"leaf": label, ...}
    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    default = majority_label(y)  # also used if a branch is missing at predict time
    n = len(y)
    h = entropy(y)  # H(S) at this node

    # Stopping rules (regularize H / end recursion)
    if y.nunique() == 1:
        return {"leaf": y.iloc[0], "n": n, "entropy": h}  # pure node
    if (
        len(attributes) == 0
        or len(y) < min_samples_leaf
        or (max_depth is not None and depth >= max_depth)
    ):
        return {"leaf": default, "n": n, "entropy": h}

    attr, ig = best_split(X, y, attributes)
    if ig <= 0:
        return {"leaf": default, "n": n, "entropy": h}  # no informative split left

    # Multiway split on observed values of attr (Connect-4: subset of {x, o, b})
    children = {}
    rest = [a for a in attributes if a != attr]  # classic ID3: use each attr once per path
    for v, part in X.groupby(attr):
        children[v] = id3_train(
            part, y.loc[part.index], rest,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            depth=depth + 1,
        )
    return {
        "attr": attr,
        "children": children,
        "default": default,
        "n": n,
        "entropy": h,
        "ig": ig,
    }


def predict_one(tree, row):
    # Walk from root to a leaf following row[attr]; unseen value → node majority
    node = tree
    while "leaf" not in node:
        child = node["children"].get(row[node["attr"]])
        if child is None:
            return node["default"]
        node = child
    return node["leaf"]


def predict(tree, X):
    # Batch wrapper around predict_one
    return [predict_one(tree, row) for _, row in X.iterrows()]


# Smoke demo on the §4 mini-slice only (full train + val tune is §7)
attrs_demo = list(FEATURE_COLS)
tree_mini = id3_train(
    mini_df[attrs_demo], mini_df[TARGET_COL], attrs_demo, max_depth=4
)
pred_mini = predict(tree_mini, mini_df[attrs_demo])
n_ok = sum(p == t for p, t in zip(pred_mini, mini_df[TARGET_COL]))
print(f"mini-slice: {n_ok}/{len(pred_mini)} correct (max_depth=4)")
print("root:", tree_mini.get("attr", tree_mini))


In [ ]:
# How — print the demo tree (all nodes with n, H, IG) and save to a text file

def tree_to_text(node, indent=""):
    # Turn the whole tree into a multi-line string (every node)
    if "leaf" in node:
        return (
            f"{indent}LEAF -> {node['leaf']}  "
            f"(n={node['n']}, H={node['entropy']:.4f})\n"
        )

    text = (
        f"{indent}[{node['attr']}]  "
        f"n={node['n']}, H={node['entropy']:.4f}, IG={node['ig']:.4f}, "
        f"majority={node['default']}\n"
    )
    for value, child in node["children"].items():
        text += f"{indent}  |-- {node['attr']} = {value}\n"
        text += tree_to_text(child, indent + "  |   ")
    return text


def print_tree(node):
    # Print every node to the notebook output
    print(tree_to_text(node), end="")


def save_tree(node, path="id3_mini_tree.txt"):
    # Write the same text view to a file (for the journal / later inspection)
    text = tree_to_text(node)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"saved tree to {path}")


print("=== full tree (n, entropy H, information gain IG) ===")
print_tree(tree_mini)
save_tree(tree_mini, "id3_mini_tree.txt")


In [ ]:
# How — visualize the demo tree with matplotlib (structure for presentation)
# Layout: leaves spaced left→right; parents centered over children; depth = row.
# Requires tree_mini from the §5 train cell (and n/H/IG fields on nodes).

def _tree_layout(node, depth=0, x_cursor=None, positions=None, edges=None):
    # Assign (x, y) to every node; y = -depth. Returns mean x of this subtree.
    if x_cursor is None:
        x_cursor = [0.0]
    if positions is None:
        positions = {}
    if edges is None:
        edges = []

    if "leaf" in node:
        x = x_cursor[0]
        x_cursor[0] += 1.0
        positions[id(node)] = (x, -float(depth), node)
        return x, positions, edges

    child_xs = []
    for value, child in node["children"].items():
        cx, positions, edges = _tree_layout(child, depth + 1, x_cursor, positions, edges)
        child_xs.append(cx)
        edges.append((id(node), id(child), str(value)))

    x = float(sum(child_xs) / len(child_xs))
    positions[id(node)] = (x, -float(depth), node)
    return x, positions, edges


def _node_label(node):
    # Short text for the box: attr or leaf class + n / H / IG
    if "leaf" in node:
        return (
            f"LEAF: {node['leaf']}\n"
            f"n={node.get('n', '?')}\n"
            f"H={node.get('entropy', float('nan')):.3f}"
        )
    return (
        f"[{node['attr']}]\n"
        f"n={node.get('n', '?')}\n"
        f"H={node.get('entropy', float('nan')):.3f}\n"
        f"IG={node.get('ig', float('nan')):.3f}"
    )


def plot_tree(node, title="ID3 tree (mini-slice)", save_path="id3_mini_tree.png"):
    # Draw decision nodes (steelblue) and leaves (darkorange); edge labels = branch values
    _, positions, edges = _tree_layout(node)

    xs = [positions[k][0] for k in positions]
    ys = [positions[k][1] for k in positions]
    width = max(xs) - min(xs) + 1.5
    height = max(ys) - min(ys) + 1.5
    fig, ax = plt.subplots(figsize=(max(8, width * 1.8), max(4, height * 1.6)))

    # Edges first (under boxes)
    for pid, cid, value in edges:
        x0, y0, _ = positions[pid]
        x1, y1, _ = positions[cid]
        ax.plot([x0, x1], [y0 - 0.15, y1 + 0.15], color="gray", lw=1.2, zorder=1)
        ax.text(
            (x0 + x1) / 2, (y0 + y1) / 2,
            value, fontsize=8, color="dimgray",
            ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85),
            zorder=2,
        )

    # Nodes
    for _, (x, y, nd) in positions.items():
        is_leaf = "leaf" in nd
        fc = "#f4a261" if is_leaf else "#4c78a8"  # orange leaf / blue split
        ax.text(
            x, y, _node_label(nd),
            ha="center", va="center", fontsize=8, color="white", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.35", fc=fc, ec="black", lw=0.8),
            zorder=3,
        )

    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved figure to {save_path}")


plot_tree(tree_mini, title="ID3 mini-slice tree (matplotlib)", save_path="id3_mini_tree.png")


## 6. System overview

### What

End-to-end path this notebook follows from raw boards to a scored tree:

| Stage | Section | Artifact |
|-------|---------|----------|
| Load + EDA + 8k subsample | §3 | `df_full` → `df` → `train_df` (feeds §4 / §5 demos only) |
| Entropy / IG check | §4 | `entropy`, `information_gain` |
| ID3 induction + infer | §5 | `id3_train`, `predict` |
| **Fixed 65/25/10 split** of `df_full` | §6 | `pool_df`, `fixed_val_df`, `fixed_test_df` |
| **Nested stratified training sets** | §6 | `train_sets[N]` for `SAMPLE_SIZES = [8000, 16000, 24000, 32000, POOL]` |
| Grid constants | §6 | `DEPTHS`, `MIN_LEAFS` |
| Progressive search (saturation → carry forward) | §7 | `runs_df`, `sat_points` |
| Freeze best config on val | §7 | `BEST_N`, `BEST_DEPTH`, `BEST_MIN_LEAF`, `tree` |
| Test metrics + baseline (once) | §8 | accuracy, 0-1 error, log-loss, macro-F1, confusion |
| Deploy sketch | §11 | single-board `predict_one` |

Val and test are **the same boards for every N**, so any change across sample sizes comes from the training set, not from a different held-out set.


### Why

Markers (and the journal) need one place that shows how the pieces connect: data controls in §3, algorithm in §5, hyperparameter choice on **validation only** in §7, and **one** test evaluation in §8. This section is wiring, not new learning theory.

The split is built once from the **full** table so that §7 can grow the training set (8k → 16k → … → the whole 65% pool) while scoring against a fixed validation set. Nesting the training sets (8k ⊂ 16k ⊂ …) means a larger N only *adds* boards, so a change in the curves is a data-size effect rather than a resampling effect.


### How — pipeline

```text
UCI Connect-4 (df_full, 67,557 boards)
    │  §6  stratified 65 / 25 / 10 (once)
    ▼
train pool (~43.9k)        val (fixed)        test (fixed, locked until §8)
    │  nested stratified: 8k ⊂ 16k ⊂ 24k ⊂ 32k ⊂ POOL
    ▼
§7  for each N:
      id3_train(train_sets[N]; max_depth, min_samples_leaf) on the allowed grid
      score train + val (accuracy, 0-1 error, log-loss)
      saturation point (d_N, m_N)  ──►  next N searches DEPTHS[i:] × MIN_LEAFS[j:]
    │
    ▼
best config on val  →  freeze tree h (BEST_N, BEST_DEPTH, BEST_MIN_LEAF)
    │  §8
    ▼
predict(test) once → accuracy / error / log-loss / macro-F1 / confusion
    │  §11
    ▼
predict_one(single board)
```

IG is the **training** split criterion; held-out accuracy / F1 are the **task** scores (Criterion C).


In [1]:
# How — fixed 65/25/10 split of df_full + nested stratified training sets (§7+)
# §3's 8k train_df still feeds the §4 mini-slice and §5 demo; everything from §7 on uses this split.

# 1) One stratified 65/25/10 split of the FULL table (val + test fixed for every N)
pool_parts, val_parts, test_parts = [], [], []
for _, g in df_full.groupby(TARGET_COL):
    g = g.sample(frac=1.0, random_state=RANDOM_SEED)
    n = len(g)
    i, j = int(0.65 * n), int(0.90 * n)
    pool_parts.append(g.iloc[:i])
    val_parts.append(g.iloc[i:j])
    test_parts.append(g.iloc[j:])

pool_df = pd.concat(pool_parts).reset_index(drop=True)
fixed_val_df = pd.concat(val_parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
fixed_test_df = pd.concat(test_parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
POOL = len(pool_df)

# 2) Nested stratified training sets: shuffle each class once, take the first ~N * share rows
SAMPLE_SIZES = [8000, 16000, 24000, 32000, POOL]
pool_by_class = {
    c: g.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    for c, g in pool_df.groupby(TARGET_COL)
}
class_share = pool_df[TARGET_COL].value_counts(normalize=True)


def class_quota(N):
    # Per-class row counts summing to exactly N (floor, then largest fractional remainders get +1)
    raw = class_share * N
    q = np.floor(raw).astype(int)
    q[(raw - q).sort_values(ascending=False).index[: N - q.sum()]] += 1
    return q


train_sets = {}
for N in SAMPLE_SIZES:
    q = class_quota(N)
    parts = [g.iloc[: q[c]] for c, g in pool_by_class.items()]
    train_sets[N] = pd.concat(parts).sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    assert len(train_sets[N]) == N

# Nesting check: every smaller set is contained in the next one (row identity via board + label)
row_key = lambda d: set(map(tuple, d[FEATURE_COLS + [TARGET_COL]].values))
for a, b in zip(SAMPLE_SIZES, SAMPLE_SIZES[1:]):
    assert row_key(train_sets[a]) <= row_key(train_sets[b]), f"{a} not nested in {b}"


def train_xy(N):
    # (X, y) for the nested training set of size N
    d = train_sets[N]
    return d[FEATURE_COLS], d[TARGET_COL]


# 3) Grid constants for the progressive search in §7
DEPTHS = [2, 6, 8, 12, 16]
MIN_LEAFS = [5, 10, 20, 30, 50]

# Shared handles: categorical cells → class label (same schema at train and deploy)
X_val, y_val = fixed_val_df[FEATURE_COLS], fixed_val_df[TARGET_COL]
X_test, y_test = fixed_test_df[FEATURE_COLS], fixed_test_df[TARGET_COL]
attrs = list(FEATURE_COLS)

print(f"df_full={len(df_full)} → pool={POOL}, val={len(fixed_val_df)}, test={len(fixed_test_df)}")
print(f"SAMPLE_SIZES={SAMPLE_SIZES}")
print(f"DEPTHS={DEPTHS}, MIN_LEAFS={MIN_LEAFS}, attrs={len(attrs)}")

# 4) Class mix per training size + val + test
mix_counts = pd.DataFrame(
    {f"train {N:,}": train_sets[N][TARGET_COL].value_counts() for N in SAMPLE_SIZES}
    | {"val": y_val.value_counts(), "test": y_test.value_counts()}
).fillna(0).astype(int)
print(mix_counts.to_string())

mix_props = mix_counts.div(mix_counts.sum(axis=0), axis=1)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
mix_counts.T.plot(kind="bar", stacked=True, ax=axes[0], colormap="tab20", legend=False)
axes[0].set_title("Class counts per set")
axes[0].set_ylabel("boards")
mix_props.T.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab20")
axes[1].set_title("Class proportions per set (stratified → equal)")
axes[1].set_ylabel("proportion")
axes[1].legend(title="class", bbox_to_anchor=(1.02, 1), loc="upper left")
for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


NameError: name 'df_full' is not defined

## 7. Training and hyperparameters

### What

Fit ID3 on **train**, choose stopping rules on **validation only**, then freeze one tree for §8.

| Step | Action |
|------|--------|
| Grid | Small sweep over `max_depth` × `min_samples_leaf` |
| Select | Highest **validation** accuracy |
| Report | Root attribute, depth, node count |
| Freeze | Refit on train with the chosen stops → `tree` |


### Why

Depth and min-leaf size shrink the hypothesis space \(H\) so the tree does not memorize every board. Picking them on validation (not test) keeps the later §8 test score honest.


### How

1. For each `(max_depth, min_samples_leaf)`: `id3_train` on train → score train & val accuracy.
2. Keep the pair with best **val** accuracy.
3. Refit once with that pair; inspect root / size.
4. Test metrics wait for §8 (one shot).


In [ ]:
# How — §7 helpers: depth-capped routing, node class counts, predict / proba, metrics
# Builds only on §5's tree format ({"leaf", ...} or {"attr", "children", "default", ...}); §5 is untouched.
# A tree grown to depth 16 and cut at depth d predicts exactly like a tree grown with max_depth=d:
# the depth stop is the only rule that looks at depth, and a cut node's "default" is the same
# majority label the shallower tree would store in that leaf.
from collections import Counter

CLASSES = sorted(df_full[TARGET_COL].unique())  # fixed column order for probabilities / log-loss


def route_path(tree, row):
    # Every node visited from the root until a leaf or an unseen branch value (no depth cap)
    path = [tree]
    node = tree
    while "leaf" not in node:
        node = node["children"].get(row[node["attr"]])
        if node is None:
            break
        path.append(node)
    return path


def route(tree, row, max_depth=None):
    # Node where `row` stops: a leaf, the depth cap, or a branch value never seen in training
    path = route_path(tree, row)
    return path[-1] if max_depth is None else path[min(max_depth, len(path) - 1)]


def node_label(node):
    # Predicted class at a stopping node (leaf label, or the stored majority of a cut / fallback node)
    return node["leaf"] if "leaf" in node else node["default"]


def node_counts(tree, X_train, y_train, max_depth=None):
    # {id(node): Counter(class → #train rows)} for every node the training rows pass through.
    # Counts at a node do not depend on the cap, so one call with max_depth=None serves every depth.
    counts = {}
    for row, label in zip(X_train.to_dict("records"), y_train):
        path = route_path(tree, row)
        if max_depth is not None:
            path = path[: max_depth + 1]
        for node in path:
            counts.setdefault(id(node), Counter())[label] += 1
    return counts


def laplace_proba(counter, classes=CLASSES, alpha=1.0):
    # Smoothed class frequencies (c_k + alpha) / (n + alpha * K) — never exactly 0, so log-loss stays finite
    n = sum(counter.values())
    return np.array([(counter.get(c, 0) + alpha) / (n + alpha * len(classes)) for c in classes])


def predict_capped(tree, X, max_depth=None):
    # Labels from the tree cut at max_depth (None = full tree, same as §5 predict)
    return [node_label(route(tree, row, max_depth)) for row in X.to_dict("records")]


def predict_proba_capped(tree, X, max_depth, counts, classes=CLASSES, alpha=1.0):
    # (n_rows, K) Laplace-smoothed class probabilities at each row's stopping node
    return np.vstack([
        laplace_proba(counts[id(route(tree, row, max_depth))], classes, alpha)
        for row in X.to_dict("records")
    ])


def accuracy(y_true, y_pred):
    # Fraction of exact label matches (0-1 task score — not information gain)
    y_true = list(y_true)
    y_pred = list(y_pred)
    return sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)


def zero_one_error(y_true, y_pred):
    # 0-1 loss averaged over rows = 1 - accuracy
    return 1.0 - accuracy(y_true, y_pred)


def log_loss(y_true, proba, classes=CLASSES, eps=1e-15):
    # Mean -log p(true class); proba columns follow `classes`
    col = {c: k for k, c in enumerate(classes)}
    p_true = proba[np.arange(len(proba)), [col[t] for t in y_true]]
    return float(-np.mean(np.log(np.clip(p_true, eps, 1.0))))


def score_capped(tree, X, y, max_depths, counts, classes=CLASSES, alpha=1.0):
    # {d: {"acc", "err", "log_loss"}} for every cap in max_depths, walking each row only once
    y = list(y)
    paths = [route_path(tree, row) for row in X.to_dict("records")]
    out = {}
    for d in max_depths:
        stops = [p[min(d, len(p) - 1)] for p in paths]
        acc = accuracy(y, [node_label(s) for s in stops])
        proba = np.vstack([laplace_proba(counts[id(s)], classes, alpha) for s in stops])
        out[d] = {"acc": acc, "err": 1.0 - acc, "log_loss": log_loss(y, proba, classes)}
    return out


def tree_depth(node, max_depth=None):
    # Longest root→leaf path (edges), optionally cut at max_depth
    if "leaf" in node or max_depth == 0:
        return 0
    nxt = None if max_depth is None else max_depth - 1
    return 1 + max(tree_depth(ch, nxt) for ch in node["children"].values())


def tree_size(node, max_depth=None):
    # Number of nodes (splits + leaves) in the tree cut at max_depth
    if "leaf" in node or max_depth == 0:
        return 1
    nxt = None if max_depth is None else max_depth - 1
    return 1 + sum(tree_size(ch, nxt) for ch in node["children"].values())


# Sanity check on 2,000 rows of the 8k nested set: cutting a deep tree matches refitting at that depth
_Xs, _ys = (v.head(2000) for v in train_xy(SAMPLE_SIZES[0]))
_Xv, _yv = X_val.head(2000), y_val.head(2000)
_deep = id3_train(_Xs, _ys, attrs, max_depth=max(DEPTHS), min_samples_leaf=MIN_LEAFS[-1])
_counts = node_counts(_deep, _Xs, _ys)
assert all(sum(_counts[id(n)].values()) == n["n"] for n in [_deep] + list(_deep["children"].values()))
for _d in DEPTHS[:3]:
    _shallow = id3_train(_Xs, _ys, attrs, max_depth=_d, min_samples_leaf=MIN_LEAFS[-1])
    assert predict_capped(_deep, _Xv, _d) == predict(_shallow, _Xv), f"cap mismatch at depth {_d}"
    assert tree_size(_deep, _d) == tree_size(_shallow), f"size mismatch at depth {_d}"
_s = score_capped(_deep, _Xv, _yv, DEPTHS[:3], _counts)
print(f"classes={CLASSES}")
print(f"cap check ok (2,000 train rows, min_leaf={MIN_LEAFS[-1]}):",
      {d: {k: round(v, 4) for k, v in m.items()} for d, m in _s.items()})


In [ ]:
# How — §7 progressive search: saturation point per N, then carry the grid forward to the next N
# Uses train_xy / SAMPLE_SIZES / DEPTHS / MIN_LEAFS / X_val / y_val / attrs from §6, id3_train from §5,
# and node_counts / score_capped / tree_depth / tree_size from the helper cell above.
# One tree per (N, min_leaf) is grown to the deepest allowed depth; every shallower depth is scored
# by cutting prediction at that depth (exact — checked in the helper cell). Test is never touched here.
import time

TOL_ACC = 0.005   # val accuracy may sit this far below the best val accuracy for this N
TOL_LOSS = 0.01   # val log-loss may sit this far above the lowest val log-loss for this N


def saturation_point(runs_n):
    # First combo in grid order (depth ascending, then min_leaf ascending) that is within both
    # tolerances. If no combo meets both, fall back to the first combo with the best val accuracy.
    ordered = runs_n.sort_values(["max_depth", "min_samples_leaf"])
    ok = ordered[
        (ordered["val_acc"] >= ordered["val_acc"].max() - TOL_ACC)
        & (ordered["val_log_loss"] <= ordered["val_log_loss"].min() + TOL_LOSS)
    ]
    if len(ok):
        return ok.iloc[0], "acc+loss"
    return ordered.loc[ordered["val_acc"].idxmax()], "acc only"


run_rows, sat_points = [], {}
depth_grid, leaf_grid = list(DEPTHS), list(MIN_LEAFS)  # N=8000 searches the full 5x5 grid
t_search = time.perf_counter()

for N in SAMPLE_SIZES:
    X_n, y_n = train_xy(N)
    print(f"\nN={N:,}: depths={depth_grid}, min_leafs={leaf_grid}")
    for m in leaf_grid:
        t0 = time.perf_counter()
        deep = id3_train(X_n, y_n, attrs, max_depth=max(depth_grid), min_samples_leaf=m)
        fit_s = time.perf_counter() - t0
        counts = node_counts(deep, X_n, y_n)  # train class counts per node → Laplace proba for log-loss
        tr = score_capped(deep, X_n, y_n, depth_grid, counts)
        va = score_capped(deep, X_val, y_val, depth_grid, counts)
        for d in depth_grid:
            run_rows.append({
                "N": N,
                "max_depth": d,
                "min_samples_leaf": m,
                "train_acc": tr[d]["acc"],
                "val_acc": va[d]["acc"],
                "train_err": tr[d]["err"],
                "val_err": va[d]["err"],
                "train_log_loss": tr[d]["log_loss"],
                "val_log_loss": va[d]["log_loss"],
                "depth": tree_depth(deep, d),
                "nodes": tree_size(deep, d),
                "fit_seconds": fit_s,  # one deep fit shared by every depth of this (N, min_leaf)
            })
        best_d = max(depth_grid, key=lambda d: va[d]["acc"])
        print(f"  min_leaf={m:>2}: fit {fit_s:7.1f}s  nodes={tree_size(deep):>6,}  "
              f"best val acc={va[best_d]['acc']:.4f} at depth {best_d}")

    runs_n = pd.DataFrame([r for r in run_rows if r["N"] == N])
    sat, rule = saturation_point(runs_n)
    d_sat, m_sat = int(sat["max_depth"]), int(sat["min_samples_leaf"])
    sat_points[N] = {
        "max_depth": d_sat,
        "min_samples_leaf": m_sat,
        "val_acc": float(sat["val_acc"]),
        "val_log_loss": float(sat["val_log_loss"]),
        "best_val_acc": float(runs_n["val_acc"].max()),
        "best_val_log_loss": float(runs_n["val_log_loss"].min()),
        "rule": rule,
    }
    print(f"N={N:,} saturation → depth={d_sat}, min_leaf={m_sat}  "
          f"(val acc={sat['val_acc']:.4f} vs best {runs_n['val_acc'].max():.4f}; "
          f"val log-loss={sat['val_log_loss']:.4f} vs lowest {runs_n['val_log_loss'].min():.4f}; {rule})")

    # Carry forward: the next N only searches DEPTHS[i:] x MIN_LEAFS[j:] from this saturation point
    depth_grid = DEPTHS[DEPTHS.index(d_sat):]
    leaf_grid = MIN_LEAFS[MIN_LEAFS.index(m_sat):]

runs_df = pd.DataFrame(run_rows)
sat_df = pd.DataFrame.from_dict(sat_points, orient="index").rename_axis("N")

print(f"\nprogressive search done: {len(runs_df)} runs from "
      f"{runs_df.drop_duplicates(['N', 'min_samples_leaf']).shape[0]} fits "
      f"in {time.perf_counter() - t_search:.0f}s (TOL_ACC={TOL_ACC}, TOL_LOSS={TOL_LOSS})")
print("\nsaturation points:")
print(sat_df.round(4).to_string())


In [ ]:
# How — accuracy and log-loss vs depth for every N (solid = val, dashed = train, star = saturation point)
# One row per sample size; the same axes limits in every row so the N-to-N shift is visible.
# Saves the combined grid plus one figure per N into figures/.
from pathlib import Path
from matplotlib.lines import Line2D

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

leaf_colors = dict(zip(MIN_LEAFS, plt.cm.viridis(np.linspace(0.0, 0.85, len(MIN_LEAFS)))))
acc_ylim = (runs_df[["train_acc", "val_acc"]].min().min() - 0.02, 1.0)
loss_ylim = (0.0, runs_df[["train_log_loss", "val_log_loss"]].max().max() * 1.05)
style_handles = [
    Line2D([], [], color="black", label="val"),
    Line2D([], [], color="black", linestyle="--", alpha=0.6, label="train"),
    Line2D([], [], color="crimson", marker="*", markersize=12, markeredgecolor="black",
           linestyle="none", label="saturation"),
]


def plot_curves_for_n(N, ax_acc, ax_loss):
    # Draw one sample size: accuracy panel (left) and log-loss panel (right)
    runs_n = runs_df[runs_df["N"] == N]
    for m, g in runs_n.groupby("min_samples_leaf"):
        g = g.sort_values("max_depth")
        c = leaf_colors[m]
        ax_acc.plot(g["max_depth"], g["val_acc"], marker="o", color=c, label=f"min_leaf={m}")
        ax_acc.plot(g["max_depth"], g["train_acc"], marker="o", markersize=3, linestyle="--", color=c, alpha=0.6)
        ax_loss.plot(g["max_depth"], g["val_log_loss"], marker="o", color=c)
        ax_loss.plot(g["max_depth"], g["train_log_loss"], marker="o", markersize=3, linestyle="--", color=c, alpha=0.6)

    s = sat_points[N]
    star = dict(marker="*", markersize=16, color="crimson", markeredgecolor="black", linestyle="none", zorder=5)
    ax_acc.plot(s["max_depth"], s["val_acc"], **star)
    ax_loss.plot(s["max_depth"], s["val_log_loss"], **star)

    sat_txt = f"saturation: depth={s['max_depth']}, min_leaf={s['min_samples_leaf']}"
    ax_acc.set_title(f"N={N:,} — accuracy ({sat_txt})", fontsize=10)
    ax_loss.set_title(f"N={N:,} — log-loss", fontsize=10)
    for ax, ylabel, ylim in [(ax_acc, "accuracy", acc_ylim), (ax_loss, "log-loss", loss_ylim)]:
        ax.set_xticks(DEPTHS)
        ax.set_xlabel("max_depth")
        ax.set_ylabel(ylabel)
        ax.set_ylim(*ylim)
        ax.grid(alpha=0.3)
    ax_acc.legend(fontsize=7, loc="best", title="val lines", title_fontsize=7)
    ax_loss.legend(handles=style_handles, fontsize=7, loc="upper left")


# 1) Combined grid: one row per N
fig, axes = plt.subplots(len(SAMPLE_SIZES), 2, figsize=(13, 3.4 * len(SAMPLE_SIZES)), squeeze=False)
for row, N in enumerate(SAMPLE_SIZES):
    plot_curves_for_n(N, axes[row, 0], axes[row, 1])
fig.suptitle("ID3 progressive search: train (dashed) vs val (solid) by depth and min_samples_leaf", y=1.0)
plt.tight_layout()
grid_path = FIG_DIR / "fig_34_progressive_curves.png"
fig.savefig(grid_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {grid_path}")

# 2) One file per sample size (same drawing, for the report / journal)
for N in SAMPLE_SIZES:
    fig_n, (ax_a, ax_l) = plt.subplots(1, 2, figsize=(13, 3.8))
    plot_curves_for_n(N, ax_a, ax_l)
    plt.tight_layout()
    path_n = FIG_DIR / f"fig_34_curves_N{N}.png"
    fig_n.savefig(path_n, dpi=150, bbox_inches="tight")
    plt.close(fig_n)
    print(f"saved {path_n}")


In [ ]:
# How — learning curve: best val accuracy / lowest val log-loss vs N, saturation point labelled at each N
# Grey dotted lines = majority-class accuracy and class-prior log-loss on val (no-model reference).
lc_df = runs_df.groupby("N").agg(
    best_val_acc=("val_acc", "max"),
    best_val_log_loss=("val_log_loss", "min"),
    runs=("val_acc", "size"),
)
lc_df["fit_seconds"] = runs_df.drop_duplicates(["N", "min_samples_leaf"]).groupby("N")["fit_seconds"].sum()
lc_df = lc_df.join(sat_df[["max_depth", "min_samples_leaf", "val_acc", "val_log_loss"]].add_prefix("sat_"))
print(lc_df.round(4).to_string())

prior = pool_df[TARGET_COL].value_counts(normalize=True)
base_val_acc = accuracy(y_val, [prior.idxmax()] * len(y_val))
base_val_loss = log_loss(list(y_val), np.tile(prior.reindex(CLASSES).values, (len(y_val), 1)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
panels = [
    (axes[0], "best_val_acc", "sat_val_acc", base_val_acc, "val accuracy", "best val accuracy"),
    (axes[1], "best_val_log_loss", "sat_val_log_loss", base_val_loss, "val log-loss", "lowest val log-loss"),
]
for ax, best_col, sat_col, base, ylabel, best_label in panels:
    ax.plot(lc_df.index, lc_df[best_col], marker="o", color="#4c78a8", label=best_label)
    ax.plot(lc_df.index, lc_df[sat_col], marker="*", markersize=12, linestyle="none",
            color="crimson", markeredgecolor="black", label="saturation point")
    ax.axhline(base, color="gray", linestyle=":", label="majority / prior baseline")
    for N, r in lc_df.iterrows():
        ax.annotate(f"d={int(r['sat_max_depth'])}, m={int(r['sat_min_samples_leaf'])}",
                    (N, r[sat_col]), textcoords="offset points", xytext=(0, 10),
                    ha="center", fontsize=8)
    ax.set_xticks(lc_df.index)
    ax.set_xticklabels([f"{n:,}" for n in lc_df.index], rotation=20)
    ax.set_xlabel("training rows N")
    ax.set_ylabel(ylabel)
    ax.margins(x=0.08, y=0.12)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_title("Learning curve: val accuracy vs N")
axes[1].set_title("Learning curve: val log-loss vs N")
plt.tight_layout()
lc_path = FIG_DIR / "fig_35_learning_curve.png"
fig.savefig(lc_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {lc_path}")
print(f"val baselines: majority acc={base_val_acc:.4f}, prior log-loss={base_val_loss:.4f}")

In [ ]:
# How — freeze: best val accuracy over every (N, depth, min_leaf) run; ties → fewer nodes, then larger N
# (then the smaller max_depth cap, since deeper caps can leave the tree unchanged).
# Sets the names §8–§11 read (BEST_N, BEST_DEPTH, BEST_MIN_LEAF, X_train, y_train, tree).
ranked = runs_df.sort_values(
    ["val_acc", "nodes", "N", "max_depth"], ascending=[False, True, False, True]
).reset_index(drop=True)
best = ranked.iloc[0]
BEST_N = int(best["N"])
BEST_DEPTH = int(best["max_depth"])
BEST_MIN_LEAF = int(best["min_samples_leaf"])
X_train, y_train = train_xy(BEST_N)

print("top 5 runs on val:")
print(ranked.head(5)[["N", "max_depth", "min_samples_leaf", "train_acc", "val_acc",
                      "val_log_loss", "nodes"]].round(4).to_string(index=False))
print(f"\nchosen on val → N={BEST_N:,}, max_depth={BEST_DEPTH}, min_samples_leaf={BEST_MIN_LEAF}")

# Refit with the chosen stops (a real max_depth tree, so §5 predict works unchanged downstream)
t0 = time.perf_counter()
tree = id3_train(X_train, y_train, attrs, max_depth=BEST_DEPTH, min_samples_leaf=BEST_MIN_LEAF)
freeze_seconds = time.perf_counter() - t0

frozen_val_acc = accuracy(y_val, predict(tree, X_val))
assert frozen_val_acc == best["val_acc"], "refit val accuracy differs from the capped search run"
assert tree_size(tree) == best["nodes"], "refit node count differs from the capped search run"
print(f"frozen tree: root={tree.get('attr', tree)}, depth={tree_depth(tree)}, nodes={tree_size(tree):,}, "
      f"fit {freeze_seconds:.1f}s, val acc={frozen_val_acc:.4f} (matches search)")

## 8. Evaluation (Criterion C)

### What

Score the **frozen** tree from §7 (`BEST_N` training rows, `BEST_DEPTH`, `BEST_MIN_LEAF`) **once** on the fixed held-out **test** set from §6. Compare to a majority-class baseline.

| Metric | Meaning here |
|--------|----------------|
| Accuracy | Overall 0–1 correct rate |
| 0-1 error | 1 − accuracy |
| Log-loss | Mean −log p(true class); p = Laplace-smoothed train class frequencies at the node where the board stops (same rule as §7) |
| Per-class P / R / F1 | Precision, recall, F1 for `win` / `loss` / `draw` |
| Macro-F1 | Unweighted mean of per-class F1 (helps under imbalance) |
| Confusion matrix | Counts of true vs predicted label |
| Test vs N | Each N's best-on-val config scored on the same test set (reporting only — nothing is re-chosen) |

**Training criterion vs task objective:** splits used information gain; we **judge** with accuracy / F1 / log-loss on boards the tree never trained on.

### Why

Criterion C needs held-out evidence, a simple baseline, and clarity that maximizing IG on train is not the same as maximizing test accuracy—especially with class imbalance (`win` is common). Log-loss adds a check that accuracy hides: whether the class frequencies in the tree's leaves are well calibrated, or confidently wrong.

### How

1. Predict on train / val / test with the frozen `tree`.
2. Majority baseline = always predict the most common **train** label; its log-loss uses the train class shares for every board.
3. Report accuracy, 0-1 error, log-loss, macro-F1 and per-class scores; plot confusion heatmap + class bars.
4. Test-vs-N curves (further down): each N's best-on-val config scored on the same test set, for reporting only.

In [ ]:
# How — test metrics + majority baseline (no sklearn; pandas/numpy only)

LABELS = sorted(y_train.unique())  # fixed order for matrices / plots


def confusion_matrix(y_true, y_pred, labels):
    # rows = true label, cols = predicted label
    idx = {c: i for i, c in enumerate(labels)}
    M = np.zeros((len(labels), len(labels)), dtype=int)
    for t, p in zip(y_true, y_pred):
        M[idx[t], idx[p]] += 1
    return M


def per_class_scores(y_true, y_pred, labels):
    # Precision / recall / F1 for each class from the confusion matrix
    M = confusion_matrix(y_true, y_pred, labels)
    rows = []
    for i, c in enumerate(labels):
        tp = M[i, i]
        fp = M[:, i].sum() - tp
        fn = M[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        rows.append({
            "class": c,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "support": int(M[i, :].sum()),
        })
    return pd.DataFrame(rows)


def macro_f1(y_true, y_pred, labels):
    return float(per_class_scores(y_true, y_pred, labels)["f1"].mean())


# Predictions from the frozen §7 tree
pred_train = predict(tree, X_train)
pred_val = predict(tree, X_val)
pred_test = predict(tree, X_test)

# Majority baseline: always the most frequent training label
majority_cls = majority_label(y_train)
base_test = [majority_cls] * len(y_test)

# Log-loss: Laplace-smoothed train class frequencies at each board's stopping node (same rule as §7);
# the baseline's log-loss gives every board the train class shares.
counts_frozen = node_counts(tree, X_train, y_train)
prior_train = y_train.value_counts(normalize=True).reindex(CLASSES).values

eval_rows = []
for split, X_s, y_s, pred_s in [
    ("train", X_train, y_train, pred_train),
    ("val", X_val, y_val, pred_val),
    ("test", X_test, y_test, pred_test),
]:
    acc = accuracy(y_s, pred_s)
    eval_rows.append({
        "model": "ID3", "split": split, "acc": acc, "err": 1.0 - acc,
        "log_loss": log_loss(list(y_s), predict_proba_capped(tree, X_s, None, counts_frozen)),
        "macro_f1": macro_f1(y_s, pred_s, LABELS),
    })
base_acc_test = accuracy(y_test, base_test)
eval_rows.append({
    "model": "majority / prior", "split": "test", "acc": base_acc_test, "err": 1.0 - base_acc_test,
    "log_loss": log_loss(list(y_test), np.tile(prior_train, (len(y_test), 1))),
    "macro_f1": macro_f1(y_test, base_test, LABELS),
})
eval_df = pd.DataFrame(eval_rows)

print(f"frozen tree: N={BEST_N:,}, max_depth={BEST_DEPTH}, min_samples_leaf={BEST_MIN_LEAF}")
print(f"majority baseline class = {majority_cls}")
print(eval_df.round(4).to_string(index=False))

cls_df = per_class_scores(y_test, pred_test, LABELS)
print("\nper-class (test, ID3):")
print(cls_df.to_string(index=False))

cm = confusion_matrix(y_test, pred_test, LABELS)
print("\nconfusion (rows=true, cols=pred):")
print(pd.DataFrame(cm, index=LABELS, columns=LABELS))

In [ ]:
# How — evaluation figures (matplotlib)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))

# 1) Accuracy: ID3 (train/val/test) vs majority baseline on test
acc_names = ["ID3 train", "ID3 val", "ID3 test", "majority test"]
acc_vals = [
    accuracy(y_train, pred_train),
    accuracy(y_val, pred_val),
    accuracy(y_test, pred_test),
    accuracy(y_test, base_test),
]
axes[0].bar(acc_names, acc_vals, color=["#4c78a8", "#4c78a8", "#4c78a8", "#f4a261"])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("accuracy")
axes[0].set_title(f"Accuracy vs majority baseline (N={BEST_N:,})")
axes[0].tick_params(axis="x", rotation=25)

# 2) Confusion heatmap on test
im = axes[1].imshow(cm, cmap="Blues")
axes[1].set_xticks(range(len(LABELS)))
axes[1].set_yticks(range(len(LABELS)))
axes[1].set_xticklabels(LABELS)
axes[1].set_yticklabels(LABELS)
axes[1].set_xlabel("predicted")
axes[1].set_ylabel("true")
axes[1].set_title("Test confusion matrix")
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        axes[1].text(j, i, str(cm[i, j]), ha="center", va="center", color="black", fontsize=9)
fig.colorbar(im, ax=axes[1], fraction=0.046)

# 3) Per-class precision / recall / F1 on test
x = np.arange(len(cls_df))
w = 0.25
axes[2].bar(x - w, cls_df["precision"], w, label="precision", color="#4c78a8")
axes[2].bar(x, cls_df["recall"], w, label="recall", color="#f4a261")
axes[2].bar(x + w, cls_df["f1"], w, label="f1", color="#6b8e23")
axes[2].set_xticks(x)
axes[2].set_xticklabels(cls_df["class"])
axes[2].set_ylim(0, 1.05)
axes[2].set_title("Per-class scores (test)")
axes[2].legend(fontsize=8)

plt.tight_layout()
eval_path = FIG_DIR / "fig_41_test_evaluation.png"
fig.savefig(eval_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {eval_path}")

### How — test score per N (reporting only)

§7 already trained on the whole training pool (`N = POOL`), so a separate full-data refit is no longer needed. Instead, for every sample size N:

1. Take that N's best run on val (same ranking as the §7 freeze: val accuracy, then fewer nodes, then the smaller depth cap).
2. Refit ID3 on the nested N-row training set with those stops (`BEST_N` reuses the frozen `tree`).
3. Score accuracy, 0-1 error and log-loss on the **same fixed test set**.

The curves show how held-out performance moves with N against the majority / prior baseline. Nothing is chosen from them: `tree` stays exactly as frozen in §7.

In [ ]:
# How — each N's best-on-val config, refit and scored on the fixed test set (reporting only)
# Ranking matches the §7 freeze; the frozen `tree` is reused for its own N instead of refitting.
test_rows, trees_by_n = [], {}
for N in SAMPLE_SIZES:
    r = runs_df[runs_df["N"] == N].sort_values(
        ["val_acc", "nodes", "max_depth"], ascending=[False, True, True]
    ).iloc[0]
    d, m = int(r["max_depth"]), int(r["min_samples_leaf"])
    X_n, y_n = train_xy(N)
    t0 = time.perf_counter()
    if (N, d, m) == (BEST_N, BEST_DEPTH, BEST_MIN_LEAF):
        t_n = tree
    else:
        t_n = id3_train(X_n, y_n, attrs, max_depth=d, min_samples_leaf=m)
    fit_s = time.perf_counter() - t0
    trees_by_n[N] = t_n

    counts_n = node_counts(t_n, X_n, y_n)
    va = score_capped(t_n, X_val, y_val, [d], counts_n)[d]
    te = score_capped(t_n, X_test, y_test, [d], counts_n)[d]
    assert va["acc"] == r["val_acc"], f"N={N}: refit val accuracy differs from the §7 run"
    test_rows.append({
        "N": N, "max_depth": d, "min_samples_leaf": m, "nodes": tree_size(t_n),
        "val_acc": va["acc"], "test_acc": te["acc"],
        "val_err": va["err"], "test_err": te["err"],
        "val_log_loss": va["log_loss"], "test_log_loss": te["log_loss"],
        "refit_seconds": fit_s,
    })
    how = "frozen tree" if t_n is tree else f"refit {fit_s:.1f}s"
    print(f"N={N:>6,}: depth={d:>2}, min_leaf={m:>2}  test acc={te['acc']:.4f}  "
          f"err={te['err']:.4f}  log-loss={te['log_loss']:.4f}  ({how})")

test_by_n_df = pd.DataFrame(test_rows).set_index("N")

# Majority / prior baseline on test: every nested set is stratified, so its class shares match the pool's
prior_pool = pool_df[TARGET_COL].value_counts(normalize=True).reindex(CLASSES)
base_test_acc = accuracy(y_test, [prior_pool.idxmax()] * len(y_test))
base_test_loss = log_loss(list(y_test), np.tile(prior_pool.values, (len(y_test), 1)))

print()
print(test_by_n_df.round(4).to_string())
print(f"\ntest baselines: majority acc={base_test_acc:.4f} (err={1 - base_test_acc:.4f}), "
      f"prior log-loss={base_test_loss:.4f}")

In [ ]:
# How — test accuracy / 0-1 error / log-loss vs N (dashed = val for the same config, star = frozen tree)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
panels = [
    (axes[0], "acc", base_test_acc, "accuracy", "majority baseline"),
    (axes[1], "err", 1 - base_test_acc, "0-1 error", "majority baseline"),
    (axes[2], "log_loss", base_test_loss, "log-loss", "class-prior baseline"),
]
for ax, metric, base, ylabel, base_label in panels:
    ax.plot(test_by_n_df.index, test_by_n_df[f"test_{metric}"], marker="o", color="#4c78a8", label="test")
    ax.plot(test_by_n_df.index, test_by_n_df[f"val_{metric}"], marker="o", markersize=3, linestyle="--",
            color="#4c78a8", alpha=0.5, label="val (same config)")
    ax.axhline(base, color="#f4a261", linestyle=":", label=base_label)
    ax.plot(BEST_N, test_by_n_df.loc[BEST_N, f"test_{metric}"], marker="*", markersize=14,
            color="crimson", markeredgecolor="black", linestyle="none", zorder=5, label="frozen tree")
    ax.set_xticks(test_by_n_df.index)
    ax.set_xticklabels([f"{n:,}" for n in test_by_n_df.index], rotation=20)
    ax.set_xlabel("training rows N")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Test {ylabel} vs N")
    ax.margins(x=0.15, y=0.12)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
for N, r in test_by_n_df.iterrows():
    axes[0].annotate(f"d={int(r['max_depth'])}, m={int(r['min_samples_leaf'])}", (N, r["test_acc"]),
                     textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)

plt.tight_layout()
test_n_path = FIG_DIR / "fig_44_test_by_n.png"
fig.savefig(test_n_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"saved {test_n_path}")

## 9. sklearn reference comparison

### What

Compare our multiway ID3 tree to an **entropy** `DecisionTreeClassifier` with matched `max_depth` / `min_samples_leaf` (from §7). sklearn is **reference only** — not the submitted model.

Both trees use the frozen §7 choice: the same `BEST_N` nested training rows (`X_train` / `y_train`), `BEST_DEPTH` and `BEST_MIN_LEAF`, scored on the fixed §6 test set.

| Check | What we report |
|-------|----------------|
| Side-by-side | Test accuracy + macro-F1 for ID3 vs sklearn |
| Disagreement | Fraction of test boards where the two models differ |
| Gap discussion | Multiway ID3 vs binary CART; categorical encoding |

### Why

The FAQ encourages an off-the-shelf reference so gaps are explained honestly (split style, feature reuse, encoding)—not “sklearn is better.”


### How

Encode `{x,o,b}` → integers for sklearn, fit with the same stops as §7, score on the same test boards, plot metric bars and a disagreement breakdown.


In [ ]:
# How — sklearn entropy tree (reference) vs our ID3 on the same test set
from sklearn.tree import DecisionTreeClassifier

# Map categorical cells to ints (sklearn needs numeric features)
VAL2INT = {"b": 0, "o": 1, "x": 2}

def encode_X(X):
    # Column-wise replace of cell values; returns int matrix
    return X.replace(VAL2INT).astype(int).values


Xtr_enc = encode_X(X_train)
Xva_enc = encode_X(X_val)
Xte_enc = encode_X(X_test)

# Matched stopping rules from §7 validation choice
sk_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=BEST_DEPTH,
    min_samples_leaf=BEST_MIN_LEAF,
    random_state=RANDOM_SEED,
)
sk_tree.fit(Xtr_enc, y_train)

# Predictions (ID3 tree already frozen in §7 as `tree`)
id3_test = predict(tree, X_test)
sk_test = list(sk_tree.predict(Xte_enc))

id3_acc = accuracy(y_test, id3_test)
sk_acc = accuracy(y_test, sk_test)
id3_f1 = macro_f1(y_test, id3_test, LABELS)
sk_f1 = macro_f1(y_test, sk_test, LABELS)
disagree = sum(a != b for a, b in zip(id3_test, sk_test)) / len(id3_test)

print(f"matched stops: N={BEST_N:,} train rows, max_depth={BEST_DEPTH}, min_samples_leaf={BEST_MIN_LEAF}")
print(f"ID3     test acc={id3_acc:.3f}  macro-F1={id3_f1:.3f}")
print(f"sklearn test acc={sk_acc:.3f}  macro-F1={sk_f1:.3f}")
print(f"disagreement rate on test = {disagree:.3f}")

# Where they disagree: true label counts (helps see which class drives gaps)
disagree_mask = [a != b for a, b in zip(id3_test, sk_test)]
disagree_true = pd.Series([t for t, d in zip(y_test, disagree_mask) if d])
print("disagreement by true class:")
print(disagree_true.value_counts())

In [ ]:
# How — side-by-side metrics + disagreement by true class
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

# Left: accuracy / macro-F1 bars
metrics = ["accuracy", "macro-F1"]
id3_vals = [id3_acc, id3_f1]
sk_vals = [sk_acc, sk_f1]
x = np.arange(len(metrics))
w = 0.35
axes[0].bar(x - w / 2, id3_vals, w, label="ID3 (ours)", color="#4c78a8")
axes[0].bar(x + w / 2, sk_vals, w, label="sklearn entropy", color="#f4a261")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Test metrics: ID3 vs sklearn")
axes[0].legend(fontsize=8)

# Right: among disagreements, which true classes appear
vc = disagree_true.value_counts().reindex(LABELS).fillna(0)
axes[1].bar(vc.index.astype(str), vc.values, color="#6b8e23")
axes[1].set_title(f"Disagreements by true class (rate={disagree:.3f})")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()

print(
    "Gap note: our ID3 is multiway on {x,o,b}; sklearn CART is binary on integer codes. "
    "Same depth/min_leaf does not mean the same hypothesis."
)


## 10. Depth experiments

### What

Four short probes on the ID3 pipeline (not a wide hyperparameter search):

| Probe | Idea |
|-------|------|
| 1. Break IG | Pick a **random** attribute instead of \(\arg\max IG\) |
| 2. Noise / constant | Label flips + a constant feature with near-zero IG |
| 3. Runtime | Wall time vs training size \(N\) (reused from §7's `fit_seconds`) |
| 4. OOV branch | Unseen cell value at predict → node majority fallback |

Probes 1–2 refit several trees, so they run on the **8,000-row nested set** (`PROBE_N`) with the frozen `BEST_DEPTH` / `BEST_MIN_LEAF`, not on up to ~44k rows. All probes score on the fixed §6 test set.

### Why

Depth over breadth: show what happens when the splitting rule or data assumptions are stressed, and that induction cost grows with \(n\).


### How

Each probe is a small code block with one plot. Probes reuse `id3_train` / `predict` and the §7–§8 helpers. Probes 1–2 train on `train_xy(PROBE_N)`; probe 3 plots the depth-16 fit times already recorded in `runs_df`, so nothing is re-timed.

In [ ]:
# How — four probes (random splits, noise/constant feature, runtime, OOV)
# Probes 1–2 refit several trees, so they use the 8k nested set; probe 3 reuses §7's fit_seconds.
import time

PROBE_N = 8000
X_probe, y_probe = train_xy(PROBE_N)
if BEST_N == PROBE_N:
    tree_probe = tree
else:
    tree_probe = id3_train(X_probe, y_probe, attrs, max_depth=BEST_DEPTH, min_samples_leaf=BEST_MIN_LEAF)

# --- Probe 1: random attribute instead of best IG ---
def id3_train_random(X, y, attributes, max_depth=None, min_samples_leaf=1, depth=0, rng=None):
    # Same stops as id3_train, but the split attribute is random (breaks IG greediness)
    if rng is None:
        rng = np.random.default_rng(RANDOM_SEED)
    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    default = majority_label(y)
    n, h = len(y), entropy(y)

    if y.nunique() == 1:
        return {"leaf": y.iloc[0], "n": n, "entropy": h}
    if len(attributes) == 0 or len(y) < min_samples_leaf or (max_depth is not None and depth >= max_depth):
        return {"leaf": default, "n": n, "entropy": h}

    attr = attributes[int(rng.integers(0, len(attributes)))]  # random, not argmax IG
    children, rest = {}, [a for a in attributes if a != attr]
    for v, part in X.groupby(attr):
        children[v] = id3_train_random(
            part, y.loc[part.index], rest,
            max_depth=max_depth, min_samples_leaf=min_samples_leaf,
            depth=depth + 1, rng=rng,
        )
    return {"attr": attr, "children": children, "default": default, "n": n, "entropy": h, "ig": float("nan")}


tree_rand = id3_train_random(
    X_probe, y_probe, attrs, max_depth=BEST_DEPTH, min_samples_leaf=BEST_MIN_LEAF
)
id3_test_acc = accuracy(y_test, predict(tree_probe, X_test))
rand_test_acc = accuracy(y_test, predict(tree_rand, X_test))
base_acc = accuracy(y_test, [majority_label(y_probe)] * len(y_test))
print(f"Probe 1 (N={PROBE_N:,}) — random splits test acc={rand_test_acc:.3f} | ID3={id3_test_acc:.3f} | "
      f"majority={base_acc:.3f}")

# --- Probe 2: constant feature IG + label noise ---
# Constant column has no information about Y → IG should be ~0
X_const = X_probe.copy()
X_const["const"] = "b"
ig_const = information_gain(X_const["const"], y_probe)
print(f"Probe 2 — IG(S, const) = {ig_const:.6f} (expect ~0)")

noise_rates = [0.0, 0.05, 0.10, 0.20]
noise_rows = []
rng_noise = np.random.default_rng(RANDOM_SEED)
classes = list(LABELS)
for rate in noise_rates:
    y_noisy = y_probe.copy().reset_index(drop=True)
    n_flip = int(rate * len(y_noisy))
    idx = rng_noise.choice(len(y_noisy), size=n_flip, replace=False)
    for i in idx:
        # Flip to a different label
        choices = [c for c in classes if c != y_noisy.iloc[i]]
        y_noisy.iloc[i] = choices[int(rng_noise.integers(0, len(choices)))]
    t_noisy = id3_train(
        X_probe.reset_index(drop=True), y_noisy, attrs,
        max_depth=BEST_DEPTH, min_samples_leaf=BEST_MIN_LEAF,
    )
    acc_n = accuracy(y_test, predict(t_noisy, X_test))
    noise_rows.append({"noise_rate": rate, "test_acc": acc_n})
    print(f"  noise={rate:.2f} → test acc={acc_n:.3f}")
noise_df = pd.DataFrame(noise_rows)

# --- Probe 3: runtime vs N (reused from §7 — every search fit grew one max_depth=16 tree) ---
runtime_df = (
    runs_df.drop_duplicates(["N", "min_samples_leaf"])[["N", "min_samples_leaf", "fit_seconds"]]
    .sort_values(["min_samples_leaf", "N"])
    .reset_index(drop=True)
)
print("Probe 3 — §7 fit seconds (rows = N, cols = min_samples_leaf; blank = trimmed by carry-forward):")
print(runtime_df.pivot(index="N", columns="min_samples_leaf", values="fit_seconds").round(1).to_string())

# --- Probe 4: unseen category at prediction (OOV) ---
# Take one test row and set a cell to a value never seen in training
row = X_test.iloc[0].copy()
oov_attr = FEATURE_COLS[0]
row[oov_attr] = "Z"  # not in {x, o, b}
oov_pred = predict_one(tree, row)
print(f"Probe 4 — OOV value at {oov_attr}='Z' → predict_one={oov_pred} (node majority fallback)")

In [ ]:
# How — probe plots (random vs ID3, noise curve, runtime vs N)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))

# 1) Random-split vs ID3 vs majority
axes[0].bar(
    ["majority", "random ID3", "IG ID3"],
    [base_acc, rand_test_acc, id3_test_acc],
    color=["#999999", "#f4a261", "#4c78a8"],
)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("test accuracy")
axes[0].set_title(f"Probe 1: break information gain (N={PROBE_N:,})")

# 2) Label noise vs test accuracy
axes[1].plot(noise_df["noise_rate"], noise_df["test_acc"], marker="o", color="#4c78a8")
axes[1].set_xlabel("train label noise rate")
axes[1].set_ylabel("test accuracy")
axes[1].set_title(f"Probe 2: label noise (N={PROBE_N:,})")
axes[1].set_ylim(0, 1.05)

# 3) Runtime vs sample size, one line per min_samples_leaf (leaf_colors from §7)
for m, g in runtime_df.groupby("min_samples_leaf"):
    axes[2].plot(g["N"], g["fit_seconds"], marker="s", color=leaf_colors[m], label=f"min_leaf={m}")
axes[2].set_xlabel("training rows N")
axes[2].set_ylabel("fit time (s), depth-16 tree")
axes[2].set_title("Probe 3: runtime vs N (from §7)")
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.show()
print(f"Probe 2 constant-feature IG was {ig_const:.6f}; Probe 4 OOV prediction was '{oov_pred}'.")

## 11. Deployment sketch

### What

Clear I/O for one board: **42 cells in** → **`{win, loss, draw}` out** via `predict_one`.

| Topic | Policy in this notebook |
|-------|-------------------------|
| Input | One row with columns `a1`…`g6`, values in `{x, o, b}` |
| Output | Predicted class from frozen `tree` |
| OOV / bad value | Follow existing branch if possible; else use that node’s **majority** (`default`) |
| Latency | Time many `predict_one` calls; report mean ms / board |


### Why

Markers need a deployment-shaped demo: same schema as training, an explicit OOV policy, and a rough sense of interactive latency.


### How

Pick one test board, render it as a 6×7 grid, run `predict_one`, show OOV behaviour, and plot a small latency histogram.


In [ ]:
# How — single-board deploy demo + OOV policy + latency
import time

BOARD_COLS = list("abcdefg")
BOARD_ROWS = list("123456")


def board_matrix(row):
    # Build 6x7 display grid (row 6 on top) from a feature row
    M = np.empty((6, 7), dtype=object)
    for r_i, r in enumerate(BOARD_ROWS):
        for c_i, c in enumerate(BOARD_COLS):
            M[5 - r_i, c_i] = row[f"{c}{r}"]
    return M


# One held-out board
demo_idx = 0
demo_row = X_test.iloc[demo_idx]
demo_true = y_test.iloc[demo_idx]
demo_pred = predict_one(tree, demo_row)
print(f"deploy demo: true={demo_true}  pred={demo_pred}")
print("board (top = row 6):")
print(pd.DataFrame(board_matrix(demo_row), columns=BOARD_COLS, index=list(reversed(BOARD_ROWS))))

# OOV policy demo: corrupt one cell to an unseen symbol
bad = demo_row.copy()
bad["d4"] = "?"
print(f"OOV policy: d4='?' → predict_one={predict_one(tree, bad)} (uses node majority when branch missing)")

# Latency: many single-board calls
n_reps = min(500, len(X_test))
times_ms = []
for i in range(n_reps):
    r = X_test.iloc[i]
    t0 = time.perf_counter()
    predict_one(tree, r)
    times_ms.append((time.perf_counter() - t0) * 1000)
print(f"latency over {n_reps} boards: mean={np.mean(times_ms):.3f} ms  p95={np.percentile(times_ms, 95):.3f} ms")


In [ ]:
# How — board heatmap + latency histogram
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

# Board occupancy-style view: encode b/o/x as 0/1/2 for a simple heatmap
enc = {"b": 0, "o": 1, "x": 2}
M = board_matrix(demo_row)
Z = np.vectorize(enc.get)(M)
im = axes[0].imshow(Z, cmap="Blues", vmin=0, vmax=2, aspect="auto")
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(BOARD_COLS)
axes[0].set_yticks(range(6))
axes[0].set_yticklabels(list(reversed(BOARD_ROWS)))
axes[0].set_title(f"Deploy board  true={demo_true}  pred={demo_pred}")
for i in range(6):
    for j in range(7):
        axes[0].text(j, i, M[i, j], ha="center", va="center", color="black", fontsize=9)

axes[1].hist(times_ms, bins=20, color="#4c78a8", edgecolor="white")
axes[1].axvline(np.mean(times_ms), color="#f4a261", linestyle="--", label=f"mean={np.mean(times_ms):.3f} ms")
axes[1].set_xlabel("predict_one time (ms)")
axes[1].set_ylabel("boards")
axes[1].set_title("Single-board latency")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 12. Reflection and journal hooks

### What

Closing notes for the A2 **journal PDF**: limitations of this ID3 study, plausible next steps, and prompts to document how the work was done (including AI use).


### Why

Option 1 is judged on understanding and critical evaluation, not only accuracy numbers. The journal needs explicit limits, future work, and an honest implementation log.


### How — limitations

| Limit | Why it matters here |
|-------|---------------------|
| **Greedy myopia** | Each split maximizes local IG; early choices can lock out better trees. |
| **Flat board** | 42 cells are treated as independent categoricals — no explicit geometry (threats, connects, columns). |
| **Depth instability** | Deeper trees fit train noise; val-chosen stops help but do not remove the issue. |
| **Scale** | ID3 here is pure Python; the depth-16 fits on the ~44k-row pool dominate §7 runtime. One fit per min-leaf (exact depth cut) keeps the search affordable, and the §10 probes use the 8k set. |
| **Saturation rule** | The saturation point, and so how much of the grid each larger N searches, depends on `TOL_ACC` / `TOL_LOSS` and on walking the grid depth first, then min-leaf. Min-leaf is carried forward by **list position**: a larger `min_samples_leaf` gives a *simpler* tree, so the trimmed grid can drop small-leaf settings that might do better at larger N. |
| **IG ≠ task loss** | Training splits on entropy reduction; we score with accuracy / macro-F1 under class imbalance. |

### How — future work

- **C4.5 / gain ratio** to dampen bias toward high-arity attributes (less critical with arity 3, still relevant).
- **Pruning** (post-hoc) instead of only pre-pruning via `max_depth` / `min_samples_leaf`.
- **Better features**: threat patterns, center control, connect-lengths — not raw cells only.
- **Ensembles** (bagging / random forests) if stability matters more than a single symbolic tree.
- Continuous attributes are out of scope here (Connect-4 is already categorical).

### How — journal prompts (fill in the PDF)

Use these as section headings or bullets in the journal:

1. **AI use:** Which tools assisted (e.g. Cursor)? What did you accept, reject, or rewrite after checking against Quinlan / the FAQ?
2. **Challenges:** Multiway `{x,o,b}` splits; empty / missing branches at predict; Colab time on 42 attributes; runtime of the progressive search up to the ~44k-row pool.
3. **Knowledge gaps verified:** Mini-slice entropy/IG hand check (§4); disagreement vs sklearn (§9); random-split and noise probes (§10).
4. **Main claim in one sentence:** What does the frozen ID3 tree show relative to the majority baseline on held-out boards?

## 13. Appendix

### What

Artifacts for the journal’s figures / “Link to Implementation”:

| Artifact | File / field |
|----------|----------------|
| Text tree (frozen `tree`, `BEST_N` rows) | `id3_frozen_tree.txt` |
| JSON tree (same) | `id3_frozen_tree.json` |
| Progressive search runs (every N × depth × min-leaf) | `progressive_runs.csv` (from `runs_df`) |
| Colab URL | Placeholder below for the public notebook link |

### Why

Markers and the journal need exportable tree views and a stable public Colab link without digging through notebook outputs.


### How

Run the export cell after §7 (it needs `tree` and `runs_df`). Paste your public Colab URL into the markdown placeholder.

In [ ]:
# How — export frozen tree(s) to text + JSON for the journal
import json
from pathlib import Path

# Prefer tree_to_text from §5 if already defined; otherwise define a minimal printer
if "tree_to_text" not in dir():
    def tree_to_text(node, indent=""):
        if "leaf" in node:
            return (
                f"{indent}LEAF -> {node['leaf']}  "
                f"(n={node.get('n', '?')}, H={node.get('entropy', float('nan')):.4f})\n"
            )
        text = (
            f"{indent}[{node['attr']}]  "
            f"n={node.get('n', '?')}, H={node.get('entropy', float('nan')):.4f}, "
            f"IG={node.get('ig', float('nan')):.4f}, majority={node.get('default', '?')}\n"
        )
        for value, child in node.get("children", {}).items():
            text += f"{indent}  |-- {node['attr']} = {value}\n"
            text += tree_to_text(child, indent + "  |   ")
        return text


def export_tree(node, stem):
    # Write human-readable text and a JSON dump of the nested dict
    txt_path = Path(f"{stem}.txt")
    json_path = Path(f"{stem}.json")
    txt_path.write_text(tree_to_text(node), encoding="utf-8")
    json_path.write_text(json.dumps(node, indent=2), encoding="utf-8")
    print(f"wrote {txt_path} and {json_path}")
    return txt_path, json_path


# Frozen tree from §7 (primary study model)
export_tree(tree, "id3_frozen_tree")
print(f"frozen tree (N={BEST_N:,}, max_depth={BEST_DEPTH}, min_leaf={BEST_MIN_LEAF}): "
      f"root={tree.get('attr', tree)}, depth={tree_depth(tree)}, nodes={tree_size(tree)}")

# Every §7 progressive-search run: N, depth, min_leaf, train/val accuracy, error, log-loss, nodes, fit time
runs_path = Path("progressive_runs.csv")
runs_df.to_csv(runs_path, index=False)
print(f"wrote {runs_path} ({len(runs_df)} runs)")

# Short preview of the frozen tree text (first lines)
preview = Path("id3_frozen_tree.txt").read_text(encoding="utf-8").splitlines()[:20]
print("\n--- preview id3_frozen_tree.txt (first lines) ---")
print("\n".join(preview))

### Colab link (journal “Link to Implementation”)

Paste the **public** Colab URL after you publish the notebook:

| Field | Value |
|-------|--------|
| Colab URL | *`https://colab.research.google.com/drive/YOUR_NOTEBOOK_ID`* |
| Repo / backup (optional) | *`…`* |

**Checklist before submitting:** Runtime → Run all succeeds on a fresh session; `ucimlrepo` + allowed libraries install; sklearn used only in §9; student name/ID filled in §0.
